In [ ]:
!pip install evaluate datasets mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
from google.colab import files, drive
import torch
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import shutil
import gc
import os
import sys
import json
import numpy as np
import nltk
import logging
from typing import List, Tuple
from pathlib import Path
import zipfile
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer, pipeline
from evaluate import load as load_metric
from datasets import Dataset
from peft import get_peft_model, LoraConfig, TaskType
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from google.cloud import storage

In [ ]:
sys.path.append('/root')

In [ ]:
from read_input_files import process_video

In [ ]:
def setup_gpu_environment():
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

    device = torch.device("cuda")
    # print(f"Using GPU: {torch.cuda.get_device_name[0]}")
    print(f"Memory usage:")
    print(f"Allocated: {torch.cuda.memory_allocated(0)/1024**2:.2f}MB")
    print(f"Cached: {torch.cuda.memory_reserved(0)/1024**2:.2f}MB")

    #Set memory efficient options
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    return device
  else:
    print("No GPU available, using CPU")
    return torch.device("cpu")

In [ ]:
def label_dataset(exercise_type, joint_positions, timestamps):
    print(f"Debug: Type of joint_positions: {type(joint_positions)}")
    if isinstance(joint_positions, list) and joint_positions:
        print(f"Debug: First element type: {type(joint_positions[0])}")

    if not isinstance(joint_positions, list) or not joint_positions:
        print(f"Skipping {exercise_type}: No valid joint positions")
        return []
    try:
        # Convert joint positions into a proper 2D NumPy array
        joint_positions_flattened = np.array([
            [joint.x, joint.y, joint.z, joint.visibility] for frame in joint_positions for joint in frame
        ])
        print(f"joint_positions_flattened.shape: {joint_positions_flattened.shape}")
        # print(f"joint_positions_flattened: {joint_positions_flattened}")

    except Exception as e:
        print(f"Error processing joint_positions: {e}")
        return []

    # print(f"Joint Positions Shape: {joint_positions_flattened.shape}")  # Debugging


    #Normalize the data
    scaler = StandardScaler()
    joint_positions_normalized = scaler.fit_transform(joint_positions_flattened)


    #k=Start with 2 clusters representing good/bad form
    k=2
    kmeans = KMeans(n_clusters=k,random_state=42, n_init="auto")
    labels = kmeans.fit_predict(joint_positions_normalized)

    if isinstance(timestamps, np.ndarray):
        timestamps = timestamps.tolist()

    print(f"Types - joint_positions_flattened: {type(joint_positions_flattened)}, timestamps: {type(timestamps)}")

    labeled_data = []
    for i in range(len(joint_positions_flattened)):
        try:
            joint_pos = joint_positions_flattened[i]
            if isinstance(joint_pos, np.ndarray):
                joint_pos = joint_pos.tolist()

            timestamp = timestamps[i] if i < len(timestamps) else None
            if isinstance(timestamp, np.generic):
                timestamp = timestamp.item()

            data_point = {
                "exercise" : str(exercise_type),
                "joint_positions": joint_pos,
                "timestamps": timestamp,
                "label": "Good form" if labels[i] == 0 else "Bad form"
            }

            json.dumps(data_point)
            labeled_data.append(data_point)
        except Exception as e:
            print(f"Error processing data point {i}: {e}")
            print(f"Types - joint_pos: {type(joint_pos)}, timestamp: {type(timestamp)}")
            continue


    return labeled_data

In [ ]:
def create_dataset(input_bucket, output_bucket):  # Pass bucket objects directly
    """Creates a labeled dataset from videos in GCS buckets.

    Args:
        input_bucket: The GCS bucket object for input videos.
        output_bucket: The GCS bucket object for output data.
    """
    print("-------CREATING DATASET------")
    exercises = {}

    # 1. No need to initialize GCS client here; it's already done in main()

    # 2. List input videos from GCS (using bucket object)
    blobs = input_bucket.list_blobs() # List all blobs in the bucket
    exercise_names = set()
    for blob in blobs:
      exercise_name = blob.name.split("/")[0] # Extract exercise name (directory)
      exercise_names.add(exercise_name)

    exercise_directory = list(exercise_names) # Convert Set to List
    print(f"exercise_names: {exercise_names}")
    print(f"exercise_directory: {exercise_directory}")

    for exercise_type in exercise_directory:
        exercise_input_prefix = exercise_type + "/" # Add exercise_type as prefix
        exercise_output_prefix = exercise_type + "/" # Add exercise_type as prefix

        # Create output directory in GCS (no direct equivalent to os.makedirs, create an empty blob if needed)
        output_blob = output_bucket.blob(exercise_output_prefix)
        output_blob.upload_from_string("") # Creates a folder-like structure

        video_blobs = input_bucket.list_blobs(prefix=exercise_input_prefix)
        video_files = [blob.name.split("/")[-1] for blob in video_blobs if blob.name.endswith(('.mp4', '.MOV'))] # Get video file names

        print(f"video_files: {video_files}")
        for video_file in video_files:
            print(f"Processing video: {video_file}")
            input_video_blob = input_bucket.blob(exercise_input_prefix + video_file)
            input_video_path = f"gs://{input_bucket.name}/{input_video_blob.name}" # Create GCS URI for input

            output_video_blob = output_bucket.blob(exercise_output_prefix + video_file)
            output_video_path = f"gs://{output_bucket.name}/{output_video_blob.name}" # Create GCS URI for output

            timestamps, joint_positions = process_video(input_video_path, output_video_path, exercise_type=exercise_type) # Your video processing function
            print(f"timestamps: {timestamps}")
            print(f"joint_positions: {joint_positions}")

            try:
                labeled_data = label_dataset(exercise_type, joint_positions, timestamps)  # Your labeling function
                print(f"labeled_data: {labeled_data}")
                if exercise_type not in exercises:
                    exercises[exercise_type] = []
                exercises[exercise_type].extend(labeled_data)
            except Exception as e:
                print(f"Error processing video {video_file}: {e}")
                continue

    try:
        json_str = json.dumps(exercises, indent=4)
        labeled_dataset_blob = output_bucket.blob("labeled_dataset.txt") # Save to GCS
        labeled_dataset_blob.upload_from_string(json_str)
        labeled_dataset = f"gs://{output_bucket.name}/{labeled_dataset_blob.name}" # Return GCS URI
        print(f"Dataset saved to {labeled_dataset}")
        return labeled_dataset
    except TypeError as e:
        # ... (rest of the error handling code)
        print(f"Error creating labeled_dataset: {e}")

In [ ]:
def load_and_split_dataset(dataset_path, test_size=.2):
    with open(dataset_path, "r") as f:
        data = json.load(f)

    all_samples = []

    for exercise, samples in data.items():
        for sample in samples:
            input_text = f"Exercise: {sample['exercise']}\nJoint Positions: {sample['joint_positions']}\nTimestamps: {sample['timestamps']}"
            output_text = sample['label']
            all_samples.append((input_text, output_text))

    #Split into training and testing sets
    train_data, test_data = train_test_split(all_samples,test_size=test_size, random_state=42)

    return train_data, test_data

In [ ]:
def load_model_gpu_optimized(training_input_videos, training_output_videos, fitness_deepseek, model_name="deepseek-ai/deepseek-llm-7b-chat"):
  device = setup_gpu_environment()

  hf_token = os.environ.get("HF_TOKEN")

  print(f"Loading Tokenizer from {model_name}")
  tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=hf_token)

  if fitness_deepseek:
    offload_folder = os.path.join(fitness_deepseek.name, "model_offload")
    os.makedirs(offload_folder, exist_ok=True)
  else:
    offload_folder = "./model_offload"
    os.makedirs(offload_folder, exist_ok=True)

  print("Loading base model...")
  model_kwargs = {
      "trust_remote_code": True,
      "offload_folder": offload_folder,
      "low_cpu_mem_usage": True,
      "use_auth_token": hf_token
  }

  if torch.cuda.is_available():
    print("Configuring for GPU...")
    model_kwargs.update({
        "device_map": "auto",
        "torch_dtype": torch.float16,
        "max_memory": {0: "120GB"},
    })
  else:
    print("Configuring for CPU...")
    model_kwargs.update({
        "device_map": {"": torch.device("cpu")},
        "torch_dtype": torch.float32
    })

  model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

  #Enable gradient checkpointing for memory efficiency
  model.enable_input_require_grads()
  model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

   # Configure generation parameters
  model.generation_config = GenerationConfig.from_pretrained(model_name)
  model.generation_config.pad_token_id = model.generation_config.eos_token_id

  # Configure LoRA with GPU optimization
  peft_config = LoraConfig(
      task_type=TaskType.CAUSAL_LM,
      inference_mode=False,
      r=4,
      lora_alpha=16,
      lora_dropout=0.05,
      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
      bias="none"
  )

  print("Applying LoRA adapters...")
  model = get_peft_model(model, peft_config)
  print("LoRa adapters applied!")
  # Load or create dataset
  # if not os.path.exists(os.path.join(base_path, "labeled_dataset.txt")):
  #     labeled_dataset = create_dataset()
  # else:
  #     labeled_dataset = os.path.join(base_path, "labeled_dataset.txt")
  labeled_dataset = create_dataset(training_input_videos, training_output_videos)
  print(f"labeled_dataset returned: {labeled_dataset}")

  print("Loading dataset...")
  train_data, test_data = load_and_split_dataset(labeled_dataset)

  print("Preparing encodings...")
  train_encodings = tokenizer(
      [x[0] for x in train_data],
      text_target=[x[1] for x in train_data],
      padding="max_length",
      truncation=True,
      max_length=256
  )
  test_encodings = tokenizer(
      [x[0] for x in test_data],
      text_target=[x[1] for x in test_data],
      padding="max_length",
      truncation=True,
      max_length=256
  )

  train_dataset = Dataset.from_dict({
      "input_ids": train_encodings["input_ids"],
      "labels": train_encodings["labels"]
  })
  test_dataset = Dataset.from_dict({
      "input_ids": test_encodings["input_ids"],
      "labels": test_encodings["labels"]
  })

  # Configure training arguments
  training_args = TrainingArguments(
      output_dir="./deepseek_finetuned",
      evaluation_strategy="steps",
      eval_steps=100,
      save_strategy="steps",
      save_steps=100,
      per_device_train_batch_size=1,
      per_device_eval_batch_size=1,
      num_train_epochs=2,
      save_total_limit=2,
      fp16=False,  # Disable fp16
      logging_dir="./logs",
      gradient_accumulation_steps=2,
      learning_rate=1e-5,
      warmup_steps=100,
      gradient_checkpointing=True,
      optim="adamw_torch",
      max_grad_norm=0.3,
      weight_decay=0.01,
      # Disable both CUDA and MPS to force CPU usage
      no_cuda=True,
      use_mps_device=False
  )

  print("Initializing trainer...")
  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      tokenizer=tokenizer
  )

  print("Starting training...")
  trainer.train()
  return test_data, model, tokenizer, device


In [ ]:
def evaluate_model(test_samples: List[Tuple[str, str]], model, tokenizer, max_length: int = 200, batch_size: int = 8, num_workers: int = 4, device: str = None):
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)

    if device is None:
        if torch.cuda.is_available():
            device = 'cuda'
        elif torch.backends.mps.is_available():
            device = 'mps' #MPS: Metal Performance Shaders
        else:
            device = 'cpu'

    logger.info(f"Using device: {device}")

    try:
        nltk.download('punkt', quiet=True)
        metric=load_metric("rouge")
        pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, batch_size=batch_size)

        def create_batches(samples, batch_size):
            for i in range(0, len(samples), batch_size):
                yield samples[i:i + batch_size]

        #Process a single batch
        def process_batch(batch):
            inputs = [text for text, _ in batch]
            references = [truth for _, truth in batch]

            #Generate text for entire batch at once
            outputs = pipe(inputs, max_length=max_length, truncation=True, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)

            return [(out[0]["generated_text"], ref) for out, ref in zip(outputs, references)]

        predictions = []
        references = []
        batches = list(create_batches(test_samples, batch_size))

        #Process batches in parallel
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            future_to_batch = {
                executor.submit(process_batch, batch): batch for batch in batches
            }

            #Show progress bar
            with tqdm(total=len(batches), desc="Processing batches") as pbar:
                for future in as_completed(future_to_batch):
                    try:
                        results = future.result()
                        for pred, ref in results:
                            predictions.append(pred)
                            references.append(ref)
                        pbar.update(1)
                    except Exception as e:
                        logger.error(f"Batch processing error: {e}")
        if not predictions or not references:
            raise ValueError("No valid predictions or references generated")

        logger.info("Computing ROUGE scores...")
        results = metric.compute(predictions=predictions, references=references, use_aggregator=True)

        formatted_results = {
            metric: {
                'precision': scores.mid.precision,
                'recall': scores.mid.recall,
                'fmeasure': scores.mid.fmeasure
            }
            for metric, scores in results.items()
        }
        return formatted_results
    except Exception as e:
        logger.error(f"Evaluation failed: {e}")

In [ ]:
def main():
  storage_client = storage.Client()

  fitness_deepseek = storage_client.bucket("fitness-deepseek")
  training_input_videos = storage_client.bucket("training_input_videos")
  training_output_videos = storage_client.bucket("training_output_videos")


  test_data, model, tokenizer, device = load_model_gpu_optimized(training_input_videos=training_input_videos, training_output_videos=training_output_videos,fitness_deepseek=fitness_deepseek)
  print(f"Using device: {device}")
  model.save_pretrained(os.path.join(fitness_deepseek.name, "deepseek_finetuned"))
  tokenizer.save_pretrained(os.path.join(fitness_deepseek.name, "deepseek_finetuned"))

  results = evaluate_model(test_data, model, tokenizer, device=device)
  print(f"Results: {results}")


main()

NameError: name 'storage' is not defined